In [1]:
import pickle # Load refs and annotations
import json
import os
import pandas as pd
import numpy as np
import pprint
import json
import cv2
import random

from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import draw_bounding_boxes
from torchvision import models
import torchmetrics

import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw

from ipywidgets import FloatProgress
import math 
from torch.nn.modules.batchnorm import _BatchNorm
from torchvision.ops import box_convert

In [2]:
with open("./refcocog/annotations/refs(umd).p", "rb") as fp:
  refs = pickle.load(fp)

# 'annotations' will be a dict object mapping the 'annotation_id' to the 'bbox' to make search faster
with open("./refcocog/annotations/instances.json", "rb") as fp:
  data = json.load(fp)
  annotations = dict(sorted({ann["id"]: ann["bbox"] for ann in data["annotations"]}.items()))

In [3]:
def getcaption(elem):
    li = []
    for e in elem["sentences"]:
        li.append(e['raw'])
    return li

In [4]:
# class DataAugmentation():
# # This class is used to perform tranformation in order to have augmented data
    
#     def blur(self,img,bbox):
#         # Gaussian Blur the image
#         transform = v2.Compose([
#             v2.PILToTensor(),
#             Denormalize(),
#             v2.ToDtype(torch.float32, scale=True),
#             v2.GaussianBlur(kernel_size=5),
#             v2.ToTensor(),
#             v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             v2.Resize([224,224])
#         ])
#         img = transform(img)
#         return img,bbox

#     def rotate(self,img,bbox):
#         # Rotate the image
#         transform = v2.Compose([
#             v2.PILToTensor(),
#             Denormalize(),
#             v2.ToDtype(torch.float32, scale=True),
#             v2.RandomRotation(degrees=20),
#             v2.ToTensor(),
#             v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             v2.Resize([224,224])
#         ])
#         img = transform(img)
#         return img,bbox
    
#     def grayscale(self,img,bbox):
#         # Convert the image to grayscale
#         transform = v2.Compose([
#             v2.PILToTensor(),
#             Denormalize(),
#             v2.ToDtype(torch.float32, scale=True),
#             v2.Grayscale(num_output_channels=3),
#             v2.ToTensor(),
#             # since we do another manipulation we compute again the normalization 
#             # of the image even tho clip does it before
#             v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), 
#             v2.Resize([224,224])
#         ])
#         img = transform(img)
#         return img,bbox
    
#     def colorrand(self,img,bbox):
#         # Randomly change the color of the image
#         transform = v2.Compose([
#             v2.PILToTensor(),
#             Denormalize(),
#             v2.ToDtype(torch.float32, scale=True),
#             v2.ColorJitter(brightness=0.01*random.randrange(1,50), contrast=0.01*random.randrange(1,50), saturation=0.01*random.randrange(1,50), hue=0.01*random.randrange(1,50)),
#             v2.ToTensor(),
#             v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             v2.Resize([224,224])
#         ])
#         img = transform(img)
#         return img,bbox
    
#     def Hflip(self,img,bbox):
#         # flip horizontally both image and bbox
#         #img = v2.functional.hflip(img)
#         cxcywh_format = box_convert(bbox, 'xyxy', 'cxcywh')

#         # flip the y coordinate
#         transform = v2.Compose([
#             v2.PILToTensor(),
#             #Denormalize(),
#             v2.ToDtype(torch.float32, scale=True),
#             v2.RandomHorizontalFlip(p=1),
#             v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
#             v2.Resize([224,224])
#         ])
#         img = transform(img)
#         cxcywh_format[0] = img.shape[2] - cxcywh_format[0]

#         bbox = box_convert(cxcywh_format, 'cxcywh', 'xyxy')
#         return img, bbox

#     def random_augmentation(self,img,bbox):
#         # Randomly choose a transformation to be applied to the image
#         n = random.randint(0, 4)
#         if n == 0:
#             return DataAugmentation.blur(self,img,bbox)
#         elif n == 1:
#             return DataAugmentation.Hflip(self,img,bbox)
#         elif n == 2:
#             return DataAugmentation.grayscale(self,img,bbox)
#         elif n == 3:
#             return DataAugmentation.colorrand(self,img,bbox)
#         else:
#             return DataAugmentation.Hflip(self,img,bbox)

In [5]:
class RefCOCOG_noproc(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, model, preprocess, annotations, split="train", device = 'cpu', count = 4):
        
        self.clip_model, self.clip_preprocess = model, preprocess
        self.device = device
        #self.images = []
        self.texts = []
        self.filepaths =[]
        self.gt = []
        self.cls = []
        #self.AUG = DataAugmentation()


        temp = 0
        for elem in [d for d in refs if d["split"]==split]:
            
            # Retrieve Single image
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            #image = Image.open(file_name)

            # Retrieve possible ground truth measures
            cls = elem['category_id']
            gt = annotations[elem['ann_id']]
            bbox_tnsor = torch.tensor(gt, device=self.device)
            new_bbox = box_convert(bbox_tnsor, 'xywh', 'xyxy')
            # Get all texts related to the picture
            sentences = elem['sentences']
            # for i in sentences:
            self.texts.append(clip.tokenize(sentences[0]['raw']))
            #self.images.append(self.clip_preprocess(image))
            self.gt.append(new_bbox)
            self.cls.append(cls)
            self.filepaths.append(file_name)
            temp += 1
            if (temp > count):
                break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        #images = self.images[idx]
        gt = self.gt[idx]
        cls = self.cls[idx]
        filename = self.filepaths[idx]
        image = Image.open(filename)
        #self.clip_preprocess(image)
        # Apply augmentations
        image = self.clip_preprocess(image).to("cpu")
        #image, gt = self.AUG.random_augmentation(image,gt)
        #image = self.clip_preprocess(image).to(device)
        return text, gt, cls, filename, image

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [6]:
clip_model, clip_preprocess = clip.load("RN50", device="cpu")

In [7]:
# create dataset and dataloader
print("----------------------Processing train split----------------------------")
dataset_train = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="train")
dataloader_train = DataLoader(dataset_train, batch_size=1)
len_train = len(dataset_train)
print(f"Numero esempi in train = {len_train}")

----------------------Processing train split----------------------------
Numero esempi in train = 5


In [8]:
gts = None
clss = None
images = None

In [9]:

for _,data in enumerate(dataloader_train):
    
    #images = data[1].to(device)
    gts = data[1]
    clss = data[2]
    images = data[4]
    break

In [10]:
clss.shape

torch.Size([1])

In [11]:
# import torch
# image = torch.zeros((1, 3, 224, 224)).float()
# bbox = torch.FloatTensor([[20, 30, 100, 200], [50, 100, 150, 200]]) # [y1, x1, y2, x2] format
# labels = torch.LongTensor([6, 8]) # 0 represents background
# sub_sample = 16

In [12]:
import torch
image = images
bbox = gts # [y1, x1, y2, x2] format
labels = clss
sub_sample = 16

In [13]:
import torchvision
dummy_img = torch.zeros((1, 3, 224, 224)).float()
print(dummy_img)
#Out: torch.Size([1, 3, 800, 800])

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]])


In [14]:
model = torchvision.models.resnet50(pretrained=True)

/opt/anaconda3/envs/bagigio/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/bagigio/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [15]:
fe = list(model.children())

In [16]:
req_features = []
fee = []
k = image.clone()
for i in fe:
    k = i(k)
    if k.size()[2] < 224//16:
        break
    req_features.append(i)
    out_channels = k.size()[1]
print(len(req_features)) #30
print(out_channels) # 512

7
1024


In [17]:
faster_rcnn_fe_extractor = nn.Sequential(*req_features)

In [18]:
out_map = faster_rcnn_fe_extractor(image)
print(out_map.size())

torch.Size([1, 1024, 14, 14])


In [19]:
224/14

16.0

In [20]:
ratios = [0.5, 1, 2]
anchor_scales = [8, 16, 32]
anchor_base = np.zeros((len(ratios) * len(anchor_scales), 4), dtype=np.float32)
print(anchor_base)
print(anchor_base.shape)

[[          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]]
(9, 4)


In [21]:
ctr_y = sub_sample / 2.
ctr_x = sub_sample / 2.
print(ctr_y, ctr_x)
# Out: (8, 8)
for i in range(len(ratios)):
  for j in range(len(anchor_scales)):
    h = sub_sample * anchor_scales[j] * np.sqrt(ratios[i])
    w = sub_sample * anchor_scales[j] * np.sqrt(1./ ratios[i])
    index = i * len(anchor_scales) + j
    anchor_base[index, 0] = ctr_x - w / 2.
    anchor_base[index, 1] = ctr_y - h / 2.
    anchor_base[index, 2] = ctr_x + w / 2.
    anchor_base[index, 3] = ctr_y + h / 2.

print(anchor_base)
print(anchor_base.shape)

8.0 8.0
[[     -82.51     -37.255       98.51      53.255]
 [    -173.02      -82.51      189.02       98.51]
 [    -354.04     -173.02      370.04      189.02]
 [        -56         -56          72          72]
 [       -120        -120         136         136]
 [       -248        -248         264         264]
 [    -37.255      -82.51      53.255       98.51]
 [     -82.51     -173.02       98.51      189.02]
 [    -173.02     -354.04      189.02      370.04]]
(9, 4)


In [22]:
fe_size = (224//16)
ctr_x = np.arange(16, (fe_size+1) * 16, 16)
ctr_y = np.arange(16, (fe_size+1) * 16, 16)

In [23]:
ctr_x

array([ 16,  32,  48,  64,  80,  96, 112, 128, 144, 160, 176, 192, 208, 224])

In [24]:
index = 0
ctr = np.zeros((len(ctr_x) * len(ctr_y), 2))

for x in range(len(ctr_x)):
    for y in range(len(ctr_y)):
        ctr[index, 0] = ctr_x[x] - 8
        ctr[index, 1] = ctr_y[y] - 8
        index +=1

In [25]:
len(ctr) # number of anchors' centers in the featmap 14x14

196

In [26]:
fe_size # grandezza featmap 

14

In [27]:
sub_sample # scaling factor (from 224 -> 14)

16

In [28]:
anchors = np.zeros((fe_size * fe_size * 9, 4))
index = 0
for c in ctr:
    ctr_x, ctr_y  = c
    for i in range(len(ratios)):
        for j in range(len(anchor_scales)):
            h = sub_sample * anchor_scales[j] * np.sqrt(ratios[i])
            w = sub_sample * anchor_scales[j] * np.sqrt(1./ ratios[i])
            anchors[index, 0] = ctr_x - w / 2.
            anchors[index, 1] = ctr_y - h / 2.
            anchors[index, 2] = ctr_x + w / 2.
            anchors[index, 3] = ctr_y + h / 2.

            index += 1
print(anchors.shape)
print(anchors)

(1764, 4)
[[     -82.51     -37.255       98.51      53.255]
 [    -173.02      -82.51      189.02       98.51]
 [    -354.04     -173.02      370.04      189.02]
 ...
 [     170.75      125.49      261.25      306.51]
 [     125.49      34.981      306.51      397.02]
 [     34.981     -146.04      397.02      578.04]]


In [29]:
bbox = np.asarray([[20, 30, 100, 200], [50, 100, 150, 200]], dtype=np.float32) # [y1, x1, y2, x2] format
labels = np.asarray([6, 8], dtype=np.int8) # 0 represents background

In [30]:
anchors

array([[     -82.51,     -37.255,       98.51,      53.255],
       [    -173.02,      -82.51,      189.02,       98.51],
       [    -354.04,     -173.02,      370.04,      189.02],
       ...,
       [     170.75,      125.49,      261.25,      306.51],
       [     125.49,      34.981,      306.51,      397.02],
       [     34.981,     -146.04,      397.02,      578.04]])

In [31]:
anchors[:,0]

array([     -82.51,     -173.02,     -354.04, ...,      170.75,      125.49,      34.981])

In [32]:
inside_index = np.where(
        (anchors[:, 0] >= 0) &
        (anchors[:, 1] >= 0) &
        (anchors[:, 2] <= 224) &
        (anchors[:, 3] <= 224)
    )[0]
print(inside_index.shape) # list of indexes of valid anchors

(68,)


In [33]:
label = np.empty((len(inside_index), ), dtype=np.int32)
label.fill(-1)
print(label.shape)
print(label)

(68,)
[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]


In [34]:
valid_anchor_boxes = anchors[inside_index]
print(valid_anchor_boxes.shape)

(68, 4)


In [35]:
ious = np.empty((len(valid_anchor_boxes), 2), dtype=np.float32)
ious.fill(0)
print(bbox)
for num1, i in enumerate(valid_anchor_boxes):
    xa1, ya1, xa2, ya2  = i  
    anchor_area = (ya2 - ya1) * (xa2 - xa1)
    for num2, j in enumerate(bbox):
        xb1, yb1, xb2, yb2  = j
        box_area = (yb2 - yb1) * (xb2 - xb1)
        inter_x1 = max([xb1, xa1])
        inter_y1 = max([yb1, ya1])
        inter_x2 = min([xb2, xa2])
        inter_y2 = min([yb2, ya2])
        if (inter_x1 < inter_x2) and (inter_y1 < inter_y2):
            iter_area = (inter_y2 - inter_y1) * (inter_x2 - inter_x1)
            iou = iter_area / (anchor_area+ box_area - iter_area)
        else:
            iou = 0.
        ious[num1, num2] = iou
print(ious.shape)
#print(ious)

[[         20          30         100         200]
 [         50         100         150         200]]
(68, 2)


### Case 1
find the highest iou for each gt_box and its corresponding anchor box

In [36]:
ious

array([[     0.7823,     0.22489],
       [    0.83008,      0.2411],
       [    0.39435,     0.13294],
       [    0.48259,     0.20409],
       [    0.51864,     0.28477],
       [    0.67201,     0.31737],
       [    0.51864,     0.37704],
       [    0.71037,     0.34212],
       [    0.51864,     0.48358],
       [    0.42618,     0.48358],
       [    0.36738,     0.15801],
       [    0.44766,     0.24547],
       [    0.48025,     0.34722],
       [    0.45801,     0.42496],
       [    0.48025,     0.46708],
       [    0.48064,     0.46103],
       [    0.48025,     0.61035],
       [    0.39642,     0.61035],
       [    0.23474,   0.0047788],
       [    0.30345,    0.069975],
       [    0.26922,     0.15801],
       [    0.31837,     0.14422],
       [    0.32298,     0.24547],
       [    0.31837,     0.22954],
       [    0.34433,     0.34722],
       [    0.29257,     0.47976],
       [    0.31837,      0.3286],
       [    0.34433,     0.46708],
       [    0.30532,

In [37]:
gt_argmax_ious = ious.argmax(axis=0)
print(gt_argmax_ious) # indexes of higher iou (first bbox, second bbox)
gt_max_ious = ious[gt_argmax_ious, np.arange(ious.shape[1])]
print(gt_max_ious) # valuef of higher iou (first bbox, second bbox)


[ 1 16]
[    0.83008     0.61035]


In [38]:
argmax_ious = ious.argmax(axis=1)
print(argmax_ious.shape)
print(argmax_ious)
max_ious = ious[np.arange(len(inside_index)), argmax_ious]
print(max_ious)

(68,)
[0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[     0.7823     0.83008     0.39435     0.48259     0.51864     0.67201     0.51864     0.71037     0.51864     0.48358     0.36738     0.44766     0.48025     0.45801     0.48025     0.48064     0.61035     0.61035     0.23474     0.30345     0.26922     0.31837     0.32298     0.31837     0.34722     0.47976
      0.3286     0.46708     0.52218     0.44502     0.61035     0.52218     0.61035     0.41405     0.20129     0.25816      0.1842     0.27039     0.22739     0.27039     0.31973     0.36905      0.3286     0.42709     0.39905     0.44502     0.55346     0.52218     0.55346     0.41405      0.1191     0.18166
     0.25161     0.26945     0.33038     0.28963     0.41972     0.41972    0.092415     0.13921     0.19018     0.18335     0.24594     0.19609     0.30717     0.30717     0.10819      0.1152]


In [39]:
gt_argmax_ious = np.where(ious == gt_max_ious)[0]
print(gt_argmax_ious)

[ 1 16 17 30 32]


a) The anchor/anchors with the highest Intersection-over-Union(IoU) overlap with a ground-truth-box

b) An anchor that has an IoU overlap higher than 0.7 with ground-truth box.

c) We assign a negative label to a non-positive anchor if its IoU ratio is lower than 0.3 for all ground-truth boxes.

d) Anchors that are neither positive nor negitive do not contribute to the training objective.

Using argmax_ious and max_ious we can assign labels and locations to anchor boxes which satisify [b] and [c]. Using gt_argmax_ious we can assign labels and locations to anchor boxes which satisify [a].

In [40]:
pos_iou_threshold  = 0.7
neg_iou_threshold = 0.3

In [41]:
label # all values to -1

array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], dtype=int32)

In [42]:
label[max_ious < neg_iou_threshold] = 0

In [43]:
label[gt_argmax_ious] = 1

In [44]:
label[max_ious >= pos_iou_threshold] = 1

In [45]:
label # now values are changed based on ious

array([ 1,  1, -1, -1, -1, -1, -1,  1, -1, -1, -1, -1, -1, -1, -1, -1,  1,  1,  0, -1,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1,  1, -1,  1, -1,  0,  0,  0,  0,  0,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0,  0,  0,  0, -1,  0, -1, -1,  0,  0,  0,  0,  0,  0, -1, -1,  0,  0], dtype=int32)

In [46]:
len(label)

68

Training RPN The Faster_R-CNN paper phrases as follows Each mini-batch arises from a single image that contains many positive and negitive example anchors, but this will bias towards negitive samples as they are dominate. Instead, we randomly sample 256 anchors in an image to compute the loss function of a mini-batch, where the sampled positive and negative anchors have a ratio of up to 1:1. If there are fewer than 128 positive samples in an image, we pad the mini-batch with negitive ones.. From this we can derive two variable as follows

In [47]:
pos_ratio = 0.5
#n_sample = 256  # reduce the nunber of random sample due to our lower of anchors extracted
n_sample = 50 # number of anchors: 68

In [48]:
n_pos = pos_ratio * n_sample # total positive samples

In [49]:
pos_index = np.where(label == 1)[0]
if len(pos_index) > n_pos:
    disable_index = np.random.choice(pos_index, size=(len(pos_index) - n_pos), replace=False)
    label[disable_index] = -1

In [50]:
n_neg = n_sample * np.sum(label == 1)
neg_index = np.where(label == 0)[0]
if len(neg_index) > n_neg:
    disable_index = np.random.choice(neg_index, size=(len(neg_index) - n_neg), replace = False)
    label[disable_index] = -1

t_{x} = (x - x_{a})/w_{a}
t_{y} = (y - y_{a})/h_{a}
t_{w} = log(w/ w_a)
t_{h} = log(h/ h_a)

In [51]:
max_iou_bbox = bbox[argmax_ious]
print(max_iou_bbox)

[[         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         50         100         150         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         20          30         100         200]
 [         50         100         150         200]
 [         50         100         150         200]
 [         20          30         100         200]
 [         20          30      

In [52]:
height = valid_anchor_boxes[:, 2] - valid_anchor_boxes[:, 1]
width = valid_anchor_boxes[:, 3] - valid_anchor_boxes[:, 0]
ctr_y = valid_anchor_boxes[:, 1] + 0.5 * height
ctr_x = valid_anchor_boxes[:, 0] + 0.5 * width
base_height = max_iou_bbox[:, 2] - max_iou_bbox[:, 1]
base_width = max_iou_bbox[:, 3] - max_iou_bbox[:, 0]
base_ctr_y = max_iou_bbox[:, 1] + 0.5 * base_height
base_ctr_x = max_iou_bbox[:, 0] + 0.5 * base_width

In [53]:
eps = np.finfo(height.dtype).eps
height = np.maximum(height, eps)
width = np.maximum(width, eps)
dy = (base_ctr_y - ctr_y) / height
dx = (base_ctr_x - ctr_x) / width
dh = np.log(base_height / height)
dw = np.log(base_width / width)
anchor_locs = np.vstack((dx, dy, dh, dw)).transpose()
print(len(anchor_locs))
#print(anchor_locs)



68


In [54]:
anchor_labels = np.empty((len(anchors),), dtype=label.dtype)
anchor_labels.fill(-1)
anchor_labels[inside_index] = label
print(anchor_labels.size)


1764


In [55]:
anchors.shape[1:]

(4,)

In [56]:
anchor_locations = np.empty((len(anchors),) + anchors.shape[1:], dtype=anchor_locs.dtype)
anchor_locations.fill(0)
anchor_locations[inside_index, :] = anchor_locs
print(anchor_locations.size)


7056


cnn

In [57]:
mid_channels = 512
in_channels = 1024 # depends on the output feature map. in vgg 16 it is equal to 512
n_anchor = 9 # Number of anchors at each location
conv1 = nn.Conv2d(in_channels, mid_channels, 3, 1, 1)
reg_layer = nn.Conv2d(mid_channels, n_anchor *4, 1, 1, 0)
cls_layer = nn.Conv2d(mid_channels, n_anchor *2, 1, 1, 0) ## I will be going to use softmax here. you can equally use sigmoid if u replace 2 with 1.

In [58]:
# conv sliding layer
conv1.weight.data.normal_(0, 0.01)
conv1.bias.data.zero_()
# Regression layer
reg_layer.weight.data.normal_(0, 0.01)
reg_layer.bias.data.zero_()
# classification layer
cls_layer.weight.data.normal_(0, 0.01)
cls_layer.bias.data.zero_()

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [59]:
x = conv1(out_map) # out_map is obtained in section 1
pred_anchor_locs = reg_layer(x)
pred_cls_scores = cls_layer(x)
print(pred_cls_scores.shape, pred_anchor_locs.shape)
#Out:
#torch.Size([1, 18, 50, 50]) torch.Size([1, 36, 50, 50])

torch.Size([1, 18, 14, 14]) torch.Size([1, 36, 14, 14])


pred_cls_scores and pred_anchor_locs are the output the RPN network and the losses to updates the weights

pred_cls_scores and objectness_scores are used as inputs to the proposal layer, which generate a set of proposal which are further used by RoI network. We will see this in the next section.


In [60]:
pred_anchor_locs.shape

torch.Size([1, 36, 14, 14])

In [61]:
pred_anchor_locs = pred_anchor_locs.permute(0, 2, 3, 1).contiguous().view(1, -1, 4)
print(pred_anchor_locs.shape)
#Out: torch.Size([1, 22500, 4])
pred_cls_scores = pred_cls_scores.permute(0, 2, 3, 1).contiguous()
print(pred_cls_scores.shape)
#Out torch.Size([1, 50, 50, 18])
objectness_score = pred_cls_scores.view(1, 14, 14, 9, 2)[:, :, :, :, 1].contiguous().view(1, -1)
print(objectness_score.shape)
#Out torch.Size([1, 22500])
pred_cls_scores  = pred_cls_scores.view(1, -1, 2)
print(pred_cls_scores.shape)
# Out torch.size([1, 22500, 2])


torch.Size([1, 1764, 4])
torch.Size([1, 14, 14, 18])
torch.Size([1, 1764])
torch.Size([1, 1764, 2])


The proposal function will take the following parameters

- Weather training_mode or testing mode
- nms_thresh
- n_train_pre_nms — number of bboxes before nms during training
- n_train_post_nms — number of bboxes after nms during training
- n_test_pre_nms — number of bboxes before nms during testing
- n_test_post_nms — number of bboxes after nms during testing
- min_size — minimum height of the object required to create a proposal.


The Faster R_CNN says, RPN proposals highly overlap with each other. To reduced redundancy, we adopt non-maximum supression (NMS) on the proposal regions based on their cls scores. We fix the IoU threshold for NMS at 0.7, which leaves us about 2000 proposal regions per image. After an ablation study, the authors show that NMS does not harm the ultimate detection accuracy, but substantially reduces the number of proposals. After NMS, we use the top-N ranked proposal regions for detection. In the following we training Fast R-CNN using 2000 RPN proposals. During testing they evaluate only 300 proposals, they have tested this with various numbers and obtained this.

In [62]:
nms_thresh = 0.7
n_train_pre_nms = 1765
n_train_post_nms = 200
n_test_pre_nms = 6000
n_test_post_nms = 300
min_size = 16


convert the loc predictions from the rpn network to bbox [y1, x1, y2, x2] format.

This is the reverse operations of what we have done while assigning ground truth to anchor boxes .This operation decodes predictions by un-parameterizing them and offseting to image. the formulas are as follows

x = (w_{a} * ctr_x_{p}) + ctr_x_{a}
y = (h_{a} * ctr_x_{p}) + ctr_x_{a}
h = np.exp(h_{p}) * h_{a}
w = np.exp(w_{p}) * w_{a}
and later convert to y1, x1, y2, x2 format


In [63]:
anchors.shape

(1764, 4)

In [64]:
anc_height = anchors[:, 2] - anchors[:, 1]
anc_width = anchors[:, 3] - anchors[:, 0]
anc_ctr_y = anchors[:, 1] + 0.5 * anc_height
anc_ctr_x = anchors[:, 0] + 0.5 * anc_width

In [65]:
anc_height.shape

(1764,)

In [66]:
anc_ctr_y[:, np.newaxis].shape

(1764, 1)

In [67]:
pred_anchor_locs_numpy = pred_anchor_locs[0].data.numpy()
objectness_score_numpy = objectness_score[0].data.numpy()
dy = pred_anchor_locs_numpy[:, 1::4]
print(dy.shape)
dx = pred_anchor_locs_numpy[:, 0::4]
print(dx)
dh = pred_anchor_locs_numpy[:, 2::4]
dw = pred_anchor_locs_numpy[:, 3::4]
ctr_y = dy * anc_height[:, np.newaxis] + anc_ctr_y[:, np.newaxis]
ctr_x = dx * anc_width[:, np.newaxis] + anc_ctr_x[:, np.newaxis]
h = np.exp(dh) * anc_height[:, np.newaxis]
w = np.exp(dw) * anc_width[:, np.newaxis]

(1764, 1)
[[  0.0027371]
 [   0.016168]
 [  0.0014695]
 ...
 [   0.020941]
 [  -0.030337]
 [  0.0055823]]


In [68]:
roi = np.zeros(pred_anchor_locs_numpy.shape, dtype=pred_anchor_locs_numpy.dtype)
roi[:, 1::4] = ctr_y - 0.5 * h
roi[:, 0::4] = ctr_x - 0.5 * w
roi[:, 3::4] = ctr_y + 0.5 * h
roi[:, 2::4] = ctr_x + 0.5 * w

In [69]:
img_size = (224, 224) #Image size
roi[:, slice(0, 4, 2)] = np.clip(
            roi[:, slice(0, 4, 2)], 0, img_size[0])
roi[:, slice(1, 4, 2)] = np.clip(
    roi[:, slice(1, 4, 2)], 0, img_size[1])
print(roi)

[[          0           0        55.5       97.44]
 [          0           0      106.74      192.72]
 [          0           0      188.67         224]
 ...
 [     174.05       129.1         224         224]
 [     119.57       40.24         224         224]
 [     33.918           0         224         224]]


In [70]:
hs = roi[:, 3] - roi[:, 1]
ws = roi[:, 2] - roi[:, 0]
keep = np.where((hs >= min_size) & (ws >= min_size))[0]
roi = roi[keep, :]
score = objectness_score_numpy[keep]
print(score.shape)

(1632,)


In [71]:
order = score.ravel().argsort()[::-1]
print(order)

[1514 1425  300 ...  629  315  757]


In [72]:
order = order[:n_train_pre_nms]
roi = roi[order, :]
print(roi.shape)
print(roi)

(1632, 4)
[[     28.387           0         224         224]
 [     17.631           0      131.58         224]
 [          0      68.217         224      222.52]
 ...
 [          0           0         224         224]
 [          0           0         224         224]
 [          0           0         224         224]]


Apply non-maximum supression

In [73]:
y1 = roi[:, 1]
x1 = roi[:, 0]
y2 = roi[:, 3]
x2 = roi[:, 2]
area = (x2 - x1 + 1) * (y2 - y1 + 1)
#print(area)
order = score.argsort()[::-1]
print(order.size)
#print("LEN", len(order))
keep = []
while order.size > 0:
    i = order[0]
    
    keep.append(i)

    xx1 = np.maximum(x1[i], x1[order[1:]])
    yy1 = np.maximum(y1[i], y1[order[1:]])
    xx2 = np.minimum(x2[i], x2[order[1:]])
    yy2 = np.minimum(y2[i], y2[order[1:]])
    w = np.maximum(0.0, xx2 - xx1 + 1)
    h = np.maximum(0.0, yy2 - yy1 + 1)

    inter = w * h
    ovr = inter / (area[i] + area[order[1:]] - inter)
    inds = np.where(ovr <= nms_thresh)[0]
    order = order[inds + 1]

keep = keep[:n_train_post_nms] # while training/testing , use accordingly
roi = roi[keep] # the final region proposals

1632


In [74]:
len(roi)

200

In [75]:
n_sample = 128
pos_ratio = 0.25
pos_iou_thresh = 0.5
neg_iou_thresh_hi = 0.5
neg_iou_thresh_lo = 0.0

In [76]:
ious = np.empty((len(roi), 2), dtype=np.float32)
ious.fill(0)
for num1, i in enumerate(roi):
    xa1, ya1, xa2, ya2  = i  
    anchor_area = (ya2 - ya1) * (xa2 - xa1)
    for num2, j in enumerate(bbox):
        xb1, yb1, xb2, yb2 = j
        box_area = (yb2- yb1) * (xb2 - xb1)
        inter_x1 = max([xb1, xa1])
        inter_y1 = max([yb1, ya1])
        inter_x2 = min([xb2, xa2])
        inter_y2 = min([yb2, ya2])
        if (inter_x1 < inter_x2) and (inter_y1 < inter_y2):
            iter_area = (inter_y2 - inter_y1) * (inter_x2 - inter_x1)
            iou = iter_area / (anchor_area+box_area - iter_area)            
        else:
            iou = 0.
        ious[num1, num2] = iou
print(ious.shape)
#Out:
#[1535, 2]

(200, 2)


In [77]:
gt_assignment = ious.argmax(axis=1)
max_iou = ious.max(axis=1)
print(gt_assignment)
print(max_iou)
#Out:
# [0, 0, 0 ... 1, 1, 0]
# [0.016, 0., 0. ... 0.08034518, 0.10739268, 0.]

[1 0 1 1 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 1 1 1 1 0 0 0 0 1 0 1 0 1 1 1 1 1 1 0 1 0 1 0 1 0 0 1 0 0 1 0 0 1 1 0 1 0 1 1 1 0 0 0 1 1 1 1 0 0 0 0 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 1 1 1 0 1 1 1 0 1 1 1 0 0 0 1 0 1 1 0 1 1 1 1 1 1 1 0 0 1 0 1 0 0 0 0 0 0 1 0 1 1 0 1 0 1 1 0 1 1 0 0 1 1 0 1 0 0 1 0 0 0 0 0 0 1 1 1 1 1 0 0 1 0 1
 1 1 0 1 0 0 0 1 1 1 0 1 1 1 1 1 1 1 0 1 0 1 1 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 1 1]
[    0.20686     0.28166     0.16793     0.30816     0.40491     0.17998           0     0.31015     0.23147     0.19166     0.33952     0.31078     0.44114     0.42635     0.45659      0.3014     0.11766     0.27393     0.72307     0.29861      0.5765     0.19309     0.18449           0     0.39259     0.63632
     0.34531    0.097579           0     0.36349     0.27943    0.066153     0.26036     0.25338     0.29578     0.57219     0.51357     0.64009     0.24613     0.48092     0.13646           0     0.18746     0.39397     0.50946     0.21244     0.27442     0.22076     0.18745     0.1

In [78]:
gt_roi_label = labels[gt_assignment]
print(gt_roi_label)
#Out:
#[6, 6, 6, ..., 8, 8, 6]

[8 6 8 8 6 6 6 6 6 8 8 8 6 8 6 6 6 8 6 8 8 8 8 6 6 6 6 8 6 8 6 8 8 8 8 8 8 6 8 6 8 6 8 6 6 8 6 6 8 6 6 8 8 6 8 6 8 8 8 6 6 6 8 8 8 8 6 6 6 6 8 8 6 6 8 6 8 6 6 6 8 8 6 8 6 8 8 8 6 8 8 8 6 8 8 8 6 6 6 8 6 8 8 6 8 8 8 8 8 8 8 6 6 8 6 8 6 6 6 6 6 6 8 6 8 8 6 8 6 8 8 6 8 8 6 6 8 8 6 8 6 6 8 6 6 6 6 6 6 8 8 8 8 8 6 6 8 6 8
 8 8 6 8 6 6 6 8 8 8 6 8 8 8 8 8 8 8 6 8 6 8 8 6 6 6 6 6 8 8 6 8 6 6 6 8 6 6 6 8 8]


In [79]:
pos_index.size

7

In [80]:
pos_roi_per_image = 32

In [81]:
pos_index = np.where(max_iou >= pos_iou_thresh)[0]
pos_roi_per_this_image = int(min(pos_roi_per_image, pos_index.size))
if pos_index.size > 0:
    pos_index = np.random.choice(
        pos_index, size=pos_roi_per_this_image, replace=False)
print(pos_roi_per_this_image)
print(pos_index)
#Out
# 18
# [ 257  296  317 1075 1077 1169 1213 1258 1322 1325 1351 1378 1380 1425
#  1472 1482 1489 1495]

8
[44 25 36 35 37 20 18 92]


In [82]:
neg_index = np.where((max_iou < neg_iou_thresh_hi) &
                             (max_iou >= neg_iou_thresh_lo))[0]
neg_roi_per_this_image = n_sample - pos_roi_per_this_image
neg_roi_per_this_image = int(min(neg_roi_per_this_image,
                                 neg_index.size))
if  neg_index.size > 0 :
    neg_index = np.random.choice(
        neg_index, size=neg_roi_per_this_image, replace=False)
print(neg_roi_per_this_image)
print(neg_index)
#Out:
#110
# [  79  688  160  ...  376  712 1235  148 1001]

120
[168  32  46 143 177   8 181 194  27 149  72  66  42 111  90 142 109 175  51 186  77 197 106 116 165 147 154 187 145  47 184 119 112  97 138 108  23 131 185 196  87 158  57 156 100  89  29 183 105 170  99  86 135  74 130 124 162  14 193 146  41  83 153  11 150 136 144 190  60 113 189  96 157 123 115  98 121 129  21
 169 180 188  52 182  84   9  12  65 160  64 125  33 104 161 151  10   4 137 179 134  95 122  50  45  93  61  17 128  69 118  70 107 132  15  59  40  19  63 139  13]


In [83]:
keep_index = np.append(pos_index, neg_index)
gt_roi_labels = gt_roi_label[keep_index]
gt_roi_labels[pos_roi_per_this_image:] = 0  # negative labels --> 0
sample_roi = roi[keep_index]
print(sample_roi.shape)
#Out:
#(128, 4)

(128, 4)


In [84]:
bbox_for_sampled_roi = bbox[gt_assignment[keep_index]]
print(bbox_for_sampled_roi.shape)
#Out
#(128, 4)
height = sample_roi[:, 3] - sample_roi[:, 1]
width = sample_roi[:, 2] - sample_roi[:, 0]
ctr_y = sample_roi[:, 1] + 0.5 * height
ctr_x = sample_roi[:, 0] + 0.5 * width
base_height = bbox_for_sampled_roi[:, 3] - bbox_for_sampled_roi[:, 1]
base_width = bbox_for_sampled_roi[:, 2] - bbox_for_sampled_roi[:, 0]
base_ctr_y = bbox_for_sampled_roi[:, 1] + 0.5 * base_height
base_ctr_x = bbox_for_sampled_roi[:, 0] + 0.5 * base_width

(128, 4)


In [85]:
eps = np.finfo(height.dtype).eps
height = np.maximum(height, eps)
width = np.maximum(width, eps)
dy = (base_ctr_y - ctr_y) / height
dx = (base_ctr_x - ctr_x) / width
dh = np.log(base_height / height)
dw = np.log(base_width / width)
gt_roi_locs = np.vstack((dx, dy, dh, dw)).transpose()
print(gt_roi_locs.shape)
#Out:
# [[-0.08075945, -0.14638858, -0.23822695, -0.23150307],
#  [ 0.04865225,  0.15570255,  0.08902431, -0.5969549 ],
#  [ 0.17411101,  0.2244332 ,  0.19870323,  0.25063717],
#  .....
#  [-0.13976236,  0.121031  ,  0.03863466,  0.09662855],
#  [-0.59361845, -2.5121436 ,  0.04558792,  0.9731178 ],
#  [ 0.1041566 , -0.7840459 ,  1.4283055 ,  0.95092565]]

(128, 4)


So now we have gt_roi_locs and gt_roi_labels for the sampled rois. We now need design the Fast rcnn network and predict the locs and labels, Which we will do in the next section.

In [86]:
rois = torch.from_numpy(sample_roi).float()
roi_indices = 0 * np.ones((len(rois),), dtype=np.int32)
roi_indices = torch.from_numpy(roi_indices).float()
print(rois.shape, roi_indices.shape)
#Out:
#torch.Size([128, 4]) torch.Size([128])

torch.Size([128, 4]) torch.Size([128])


concat rois and roi_indices, so that we get the tensor with shape [N, 5] (index, x, y, h, w)

In [87]:
indices_and_rois = torch.cat([roi_indices[:, None], rois], dim=1)
xy_indices_and_rois = indices_and_rois[:, [0, 2, 1, 4, 3]]
indices_and_rois = xy_indices_and_rois.contiguous()
print(xy_indices_and_rois.shape)
#Out:
#torch.Size([128, 5])

torch.Size([128, 5])


In [88]:
size = (7, 7)
adaptive_max_pool = nn.AdaptiveMaxPool2d(size)
output = []
rois = indices_and_rois.data.float()
rois[:, 1:].mul_(1/16.0) # Subsampling ratio
rois = rois.long()

num_rois = rois.size(0)
for i in range(num_rois):
    roi = rois[i]
    im_idx = roi[0]
    im = out_map.narrow(0, im_idx, 1)[..., roi[2]:(roi[4]+1), roi[1]:(roi[3]+1)]
    output.append(adaptive_max_pool(im))

output = torch.cat(output, 0)
print(output.size())
#Out:
# torch.Size([128, 512, 7, 7])
# Reshape the tensor so that we can pass it through the feed forward layer.
k = output.view(output.size(0), -1)
print(k.shape)
#Out:
# torch.Size([128, 25088])

torch.Size([128, 1024, 7, 7])
torch.Size([128, 50176])


In [89]:
roi_head_classifier = nn.Sequential(*[nn.Linear(50176, 25088), nn.Linear(25088, 4096),
                                      nn.Linear(4096, 4096)])
cls_loc = nn.Linear(4096, 81 * 4) # (VOC 20 classes + 1 background. Each will have 4 co-ordinates)
cls_loc.weight.data.normal_(0, 0.01)
cls_loc.bias.data.zero_()
score = nn.Linear(4096, 81) # (VOC 20 classes + 1 background)

In [90]:
k = roi_head_classifier(k)
roi_cls_loc = cls_loc(k)
roi_cls_score = score(k)
print(roi_cls_loc.shape, roi_cls_score.shape)
#Out:
# torch.Size([128, 84]), torch.Size([128, 21])

torch.Size([128, 324]) torch.Size([128, 81])


In [91]:
print(pred_anchor_locs.shape)
print(pred_cls_scores.shape)
print(anchor_locations.shape)
print(anchor_labels.shape)
#Out:
# torch.Size([1, 12321, 4])
# torch.Size([1, 12321, 2])
# (12321, 4)
# (12321,)

torch.Size([1, 1764, 4])
torch.Size([1, 1764, 2])
(1764, 4)
(1764,)


In [92]:
rpn_loc = pred_anchor_locs[0]
rpn_score = pred_cls_scores[0]
gt_rpn_loc = torch.from_numpy(anchor_locations)
gt_rpn_score = torch.from_numpy(anchor_labels)
print(rpn_loc.shape, rpn_score.shape, gt_rpn_loc.shape, gt_rpn_score.shape)
#Out
# torch.Size([12321, 4]) torch.Size([12321, 2]) torch.Size([12321, 4]) torch.Size([12321])

torch.Size([1764, 4]) torch.Size([1764, 2]) torch.Size([1764, 4]) torch.Size([1764])


In [93]:
import torch.nn.functional as F
rpn_cls_loss = F.cross_entropy(rpn_score, gt_rpn_score.long(), ignore_index = -1)
print(rpn_cls_loss)
#Out:
# Variable containing:
#  0.6940
# [torch.FloatTensor of size 1]

tensor(0.6956, grad_fn=<NllLossBackward0>)


In [94]:
pos = gt_rpn_score > 0
mask = pos.unsqueeze(1).expand_as(rpn_loc)
print(mask.shape)
#Out:
# torch.Size(12321, 4)

torch.Size([1764, 4])


In [95]:
mask_loc_preds = rpn_loc[mask].view(-1, 4)
mask_loc_targets = gt_rpn_loc[mask].view(-1, 4)
print(mask_loc_preds.shape, mask_loc_preds.shape)
#Out:
# torch.Size([6, 4]) torch.Size([6, 4])

torch.Size([7, 4]) torch.Size([7, 4])


In [96]:
x = torch.abs(mask_loc_targets - mask_loc_preds)
rpn_loc_loss = ((x < 1).float() * 0.5 * x**2) + ((x >= 1).float() * (x-0.5))
print(rpn_loc_loss.sum())
#Out:
# Variable containing:
#  0.3826
# [torch.FloatTensor of size 1]

tensor(0.6193, dtype=torch.float64, grad_fn=<SumBackward0>)


In [97]:
rpn_lambda = 1.
N_reg = (gt_rpn_score >0).float().sum()
print(N_reg)
rpn_loc_loss = rpn_loc_loss.sum() / N_reg
rpn_loss = rpn_cls_loss + (rpn_lambda * rpn_loc_loss)
print(rpn_loss)
#Out:0.00248

tensor(7.)
tensor(0.7840, dtype=torch.float64, grad_fn=<AddBackward0>)


In [98]:
print(roi_cls_loc.shape)
print(roi_cls_score.shape)
#Out:
# torch.Size([128, 84])
# torch.Size([128, 21])

torch.Size([128, 324])
torch.Size([128, 81])


In [99]:
print(gt_roi_locs.shape)
print(gt_roi_labels.shape)
#Out:
#(128, 4)
#(128, )

(128, 4)
(128,)


In [100]:
gt_roi_loc = torch.from_numpy(gt_roi_locs)
gt_roi_label = torch.from_numpy(np.float32(gt_roi_labels)).long()
print(gt_roi_loc.shape, gt_roi_label.shape)
#Out:
#torch.Size([128, 4]) torch.Size([128])

torch.Size([128, 4]) torch.Size([128])


In [101]:
roi_cls_score

tensor([[ 0.0282,  0.0247, -0.0048,  ...,  0.0015,  0.0037,  0.0317],
        [ 0.0077, -0.0068,  0.0025,  ..., -0.0117,  0.0209, -0.0011],
        [ 0.0527,  0.0096, -0.0254,  ...,  0.0018, -0.0052, -0.0063],
        ...,
        [ 0.0093, -0.0231, -0.0221,  ...,  0.0152, -0.0084, -0.0097],
        [ 0.0414,  0.0125, -0.0202,  ..., -0.0033,  0.0254,  0.0028],
        [ 0.0397,  0.0090, -0.0156,  ...,  0.0129,  0.0021, -0.0072]], grad_fn=<AddmmBackward0>)

In [102]:
roi_cls_loss = F.cross_entropy(roi_cls_score, gt_roi_label, ignore_index=-1)
print(roi_cls_loss)
#Out:
#Variable containing:
#  3.0458
# [torch.FloatTensor of size 1]

tensor(4.3704, grad_fn=<NllLossBackward0>)


In [103]:
n_sample = roi_cls_loc.shape[0]
roi_loc = roi_cls_loc.view(n_sample, -1, 4)
print(roi_loc.shape)
#Out:
#torch.Size([128, 21, 4])
roi_loc = roi_loc[torch.arange(0, n_sample).long(), gt_roi_label]
print(roi_loc.shape)
#Out:
#torch.Size([128, 4])

torch.Size([128, 81, 4])
torch.Size([128, 4])


In [104]:
gt_roi_loc.shape

torch.Size([128, 4])

In [105]:
REGLoss = nn.L1Loss()

roi_loc_loss = REGLoss(roi_loc, gt_roi_loc)
print(roi_loc_loss)
#Out:
#Variable containing:
#  0.1895
# [torch.FloatTensor of size 1]

tensor(0.5337, grad_fn=<MeanBackward0>)


In [106]:
roi_lambda = 1.
roi_loss = roi_cls_loss + (roi_lambda * roi_loc_loss)
print(roi_loss)
#Out:
#Variable containing:
#  4.2353
# [torch.FloatTensor of size 1]

tensor(4.9041, grad_fn=<AddBackward0>)


In [107]:
total_loss = rpn_loss + roi_loss
total_loss

tensor(5.6881, dtype=torch.float64, grad_fn=<AddBackward0>)